# Jacket discounts: Fisher's exact test on a 4 x 2 table

**Fisher's Exact Test** · a worked example from the StatNex teaching guide

**The story.** An online shop showed four groups of 40 visitors the same jacket at different discounts (none, 10%, 20%, 30%) and counted how many bought it.

**The question.** Does the purchase rate differ between the discount levels at all?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/cochran_armitage.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Discount,Bought,Did not buy,Total
No discount,6.0,34.0,40.0
10% off,10.0,30.0,40.0
20% off,15.0,25.0,40.0
30% off,21.0,19.0,40.0
Total,52.0,108.0,160.0

These cells save `freeman_halton.py`, `totals.py`: steps that no library does, written out. The analysis imports them.

In [ ]:
%%writefile freeman_halton.py
"""Fisher's exact test for a table larger than 2x2 (Freeman and Halton 1951).

scipy.stats.fisher_exact estimates the P value of an r x c table by Monte Carlo; this
computes it exactly, as R's fisher.test does. With both margins fixed, a table's
probability is prod(r_i!) prod(c_j!) / (n! prod(n_ij!)), and P sums it over every
table no more probable than the observed one (within a relative 1e-7, R's tolerance,
so tables tied in exact arithmetic are not split by rounding).
"""

import math
from itertools import combinations

import numpy as np
from scipy import special, stats

#: Past this many tables the P value is a Monte Carlo estimate, as in the app.
EXACT_TABLE_LIMIT = 2_000_000
MONTE_CARLO_TABLES = 200_000
MONTE_CARLO_SEED = 20240501


def compositions(total, parts):
    """Every way to split ``total`` into ``parts`` non-negative counts, one per row."""
    if parts == 1:
        return np.array([[total]])
    bars = np.array(list(combinations(range(total + parts - 1), parts - 1)))
    edges = np.column_stack([np.full(len(bars), -1), bars, np.full(len(bars), total + parts - 1)])
    return np.diff(edges, axis=1) - 1


def exact_p(counts):
    """The exact P value, or None when there are too many tables to list.

    The tables are built a column at a time: each partial table is a set of row totals
    still to fill and the log of its prod(n_ij!) so far.
    """
    counts = np.asarray(counts, dtype=np.int64)
    if counts.shape[0] > counts.shape[1]:
        counts = counts.T
    rows, cols = counts.sum(axis=1), counts.sum(axis=0)
    n = int(counts.sum())
    log_factorial = special.gammaln(np.arange(n + 1) + 1.0)
    observed = log_factorial[counts].sum()
    remaining = rows[None, :].copy()
    partial = np.zeros(1)
    for total in cols[:-1]:
        if math.comb(int(total) + len(rows) - 1, len(rows) - 1) > EXACT_TABLE_LIMIT:
            return None
        splits = compositions(int(total), len(rows))
        fits = (splits[None, :, :] <= remaining[:, None, :]).all(axis=2)
        state, split = np.nonzero(fits)
        if len(state) > EXACT_TABLE_LIMIT:
            return None
        remaining = remaining[state] - splits[split]
        partial = partial[state] + log_factorial[splits[split]].sum(axis=1)
    weight = partial + log_factorial[remaining].sum(axis=1)
    constant = log_factorial[rows].sum() + log_factorial[cols].sum() - log_factorial[n]
    return float(np.exp(constant - weight[weight >= observed - 1e-7]).sum())


def freeman_halton(counts):
    """(P value, how it was found): exact, or Monte Carlo from a fixed seed."""
    p = exact_p(counts)
    if p is not None:
        return p, "exact"
    result = stats.fisher_exact(
        np.asarray(counts, dtype=np.int64),
        method=stats.MonteCarloMethod(n_resamples=MONTE_CARLO_TABLES,
                                      rng=np.random.default_rng(MONTE_CARLO_SEED)))
    return float(result.pvalue), f"Monte Carlo, {MONTE_CARLO_TABLES} tables"

In [ ]:
%%writefile totals.py
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from freeman_halton import freeman_halton

pd.set_option("display.width", 120)

## The data

In [ ]:
# A table of counts as printed: one column per category of one variable, one row per
# category of the other. Rows with a blank count are skipped.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Bought", "Did not buy"]
table = data[columns].apply(pd.to_numeric, errors="coerce")
table.index = data["Discount"].astype(str)
# A total row or column (Total, or its Japanese equivalent) is a margin, not a category, so it is left out.
# Spaces, underscores, case and full-width letters are ignored when names are compared
# (totals.py).
from totals import is_total

table = table[[name for name in columns if not is_total(name)]]
table = table[[not is_total(name) for name in table.index]]
table = table[table.notna().all(axis=1)]
observed = table.to_numpy(float)

## Observed frequencies

In [ ]:
grid = np.vstack([observed, observed.sum(axis=0)])
grid = np.column_stack([grid, grid.sum(axis=1)])
observed_table = pd.DataFrame(grid, index=[*table.index, "Total"], columns=[*table.columns, "Total"])
print(observed_table)

## Fisher's exact test

In [ ]:
chi2 = stats.chi2_contingency(observed, correction=False)[0]
cramers_v = np.sqrt(chi2 / (observed.sum() * (min(observed.shape) - 1)))
if observed.shape == (2, 2):
    # The one-sided P is taken in the direction the table leans: "greater" when
    # a*d >= b*c (the diagonal holds more than independence predicts), "less" otherwise.
    (a, b), (c, d) = observed
    two_sided = stats.fisher_exact(observed).pvalue
    direction = "greater" if a * d >= b * c else "less"
    one_sided = stats.fisher_exact(observed, alternative=direction).pvalue
    test = pd.Series({"two-sided P": two_sided, "one-sided P": one_sided})
    effect = pd.Series({"Cramer's V": cramers_v,
                        "Yule's Q": (a * d - b * c) / (a * d + b * c) if a * d + b * c else np.nan})
else:
    # Larger than 2x2: the Freeman-Halton extension, two-sided only, since "the
    # direction the table leans" means nothing with more than one degree of freedom.
    p, how = freeman_halton(observed)
    test = pd.Series({f"two-sided P ({how})": p})
    effect = pd.Series({"Cramer's V": cramers_v})
print()
print(test)

## Strength of association

In [ ]:
print()
print(effect)

## What the result says

Fisher's exact test also works on a table with more than two rows. It asks a simple question: is buying related to the discount level at all? It computes the exact chance of a table at least this uneven if the discount made no difference.

Of 160 visitors, 52 bought: 6, 10, 15 and 21 in the four groups of 40. The two-sided P value is 0.0025, far below 0.05, so the purchase rates are not all the same. Cramer's V, a measure of the strength of the link from 0 to 1, is 0.30.

So the discount level matters. But this test ignores the order of the discounts. It shows that the rates differ, not that they rise step by step: that question needs a trend test.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.